# Sprint 1 — 01: Data

Sprint 1 uses the same data as Sprint 0: the **Default of Credit Card Clients**
dataset (Yeh, 2009). The download and cleaning below are carried over unchanged
from our Sprint 0 report (Butler, Robinson & Mulryan, 2026), notebook
[`01-Introduction.ipynb`](https://github.com/Oscar339/dst-sprint0/blob/577d547/report/01-Introduction.ipynb). The reasoning is
summarised here so this report stands on its own, and the full working is
linked rather than repeated.

The output is `data/processed/taiwan-clean.csv` (29,995 customers, 24 columns),
which every later notebook loads. **Run this notebook first.**

## Setup

Paths are built with `pathlib` from this notebook's location, so the report
runs the same on everyone's machine. The layout follows the unit's convention:

* `data/raw/` — downloaded once, never edited
* `data/processed/` — regenerated by the code, not committed
* `data/output/` — figures and tables the report keeps

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent            # notebooks run from report/
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUTPUT = ROOT / "data" / "output"
for d in (RAW, PROCESSED, OUTPUT):
    d.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)

## Importing the data

**Default of Credit Card Clients** (Yeh, 2009), from the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)
(doi:10.24432/C55S3H): 30,000 cardholders at a Taiwanese bank in 2005, each
labelled by whether they defaulted in October 2005 (22% did).

The file is downloaded into `data/raw/` the first time this notebook runs and is
never edited there. It is an old-format Excel `.xls`, so reading it needs the
`xlrd` package (in `requirements.txt`).

In [ ]:
from zipfile import ZipFile

URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
TAIWAN_DIR = RAW / "taiwan-credit-default"
TAIWAN_XLS = TAIWAN_DIR / "default of credit card clients.xls"

# Download and unzip only if the file isn't already in data/raw/
if not TAIWAN_XLS.exists():
    TAIWAN_DIR.mkdir(parents=True, exist_ok=True)
    zip_path, _ = urlretrieve(URL)          # to a temporary file
    with ZipFile(zip_path) as z:
        z.extractall(TAIWAN_DIR)

print("data file:", TAIWAN_XLS.relative_to(ROOT))

The spreadsheet has a title row above the real column names, so we skip it
(`header=1`) and use `ID` as the row label. The label is
`default payment next month` (1 = defaulted). `LIMIT_BAL` is the credit limit;
`PAY_0`, `PAY_2`…`PAY_6` are repayment status for the last six months;
`BILL_AMT*` and `PAY_AMT*` are the bill and payment amounts for the same months.

In [ ]:
taiwan = pd.read_excel(TAIWAN_XLS, header=1, index_col="ID")

print(taiwan.shape)                                             # expect (30000, 24)
print(taiwan["default payment next month"].value_counts())    # 1 = default
taiwan.head()

In [ ]:
# Check the file is the one Sprint 0 cleaned
assert taiwan.shape == (30000, 24), taiwan.shape
assert set(taiwan["default payment next month"]) == {0, 1}
print("Taiwan data loaded as expected.")

## Cleaning

In Sprint 0 we checked the file against its documentation — the UCI page
(Yeh, 2009) and the paper that introduced the data (Yeh & Lien, 2009, p. 2475)
— and found five places where they diverge
([Sprint 0, 01-Introduction](https://github.com/Oscar339/dst-sprint0/blob/577d547/report/01-Introduction.ipynb)):

| # | What we found | What we do |
|---|---|---|
| 1 | The file's short names skip from `PAY_0` to `PAY_2`, but no month is missing: `PAY_0`, `BILL_AMT1` and `PAY_AMT1` are all September 2005 | Rename `PAY_0` to `PAY_1`, so the month numbers line up |
| 2 | Repayment status `PAY_*` uses -2 and 0, which the documentation doesn't list (it gives -1 = paid on time, 1–9 = months late) | Keep them, and treat -2, -1 and 0 as **categories**, not amounts of delay, when modelling (Figure 1.1) |
| 3 | `EDUCATION` has codes 0, 5, 6 (345 people); only 1–4 are documented, 4 = others | Merge into 4, "others" |
| 4 | `MARRIAGE` has code 0 (54 people); only 1–3 are documented, 3 = others | Merge into 3, "others" |
| 5 | The paper reports 25,000 customers; the file has 30,000, with the same 22.12% default rate | Use all of them |

### Duplicate rows

35 pairs of rows are identical in all 24 columns. In Sprint 0 we worked out how
many identical pairs chance alone would give, modelling each column as an
independent categorical distribution within each type of account
([`Oscar/02-duplicates.ipynb`](https://github.com/Oscar339/dst-sprint0/blob/577d547/Oscar/02-duplicates.ipynb)):

* 30 pairs are "no-activity" accounts (every bill and payment 0). These need to
  match on only 7 columns, so coincidences are easy, and the test can't say
  which pairs are real. We keep them.
* 5 pairs have the same non-zero bill and payment every month. Fewer than 0.01
  such pairs are expected by chance, so they are real duplicates and we drop
  the second copy.

**This matters more in Sprint 1 than it did in Sprint 0.** Duplicates must go
*before* the train / test split, or one copy can land in each part and a model
is tested on a customer it has already seen.

In [ ]:
Y = "default payment next month"
BILLS = [c for c in taiwan.columns if c.startswith("BILL_AMT")]
PAYS  = [c for c in taiwan.columns if c.startswith("PAY_AMT")]

no_activity = (taiwan[BILLS + PAYS] == 0).all(axis=1)
same_every_month = ((taiwan[BILLS].nunique(axis=1) == 1)
                    & (taiwan[PAYS].nunique(axis=1) == 1) & ~no_activity)

# duplicated() marks the later copy of each pair, so the lower ID is kept
to_drop = taiwan.index[taiwan.duplicated() & same_every_month]
print("exact duplicate pairs:", taiwan.duplicated().sum())
print("IDs dropped:", list(to_drop))
assert len(to_drop) == 5

clean = taiwan.drop(index=to_drop)
clean.shape

Then the recoding from the table (rows 1, 3 and 4). The `PAY_*` codes are left as they are (row 2), because how to encode them is a modelling choice, not a fix to the data — each of us makes that choice for our own model.

In [ ]:
clean = clean.rename(columns={"PAY_0": "PAY_1"})

print("before:", clean["EDUCATION"].value_counts().sort_index().to_dict(),
      clean["MARRIAGE"].value_counts().sort_index().to_dict())
clean["EDUCATION"] = clean["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
clean["MARRIAGE"] = clean["MARRIAGE"].replace({0: 3})
print("after: ", clean["EDUCATION"].value_counts().sort_index().to_dict(),
      clean["MARRIAGE"].value_counts().sort_index().to_dict())

**Figure 1.1** (reproduced from Sprint 0) shows why the undocumented `PAY_*`
codes are kept, and why they can't be read as numbers. For September's status
it shows the share of customers in each code who defaulted in October. The two
undocumented codes (orange) default at the same low rate as -1, "paid on time";
the step up comes at 1 month late. A model that treats the codes as a count of
months would wrongly place 0 between -1 and 1. Codes 4–8 hold only 141
customers between them, so they are shown as one bar.

In [ ]:
status = clean["PAY_1"].clip(upper=4)          # 4-8 are rare: merge into one bar
by_code = clean.groupby(status)[Y].agg(rate="mean", n="size")
labels = [f"{k if k < 4 else '4–8'}\nn={n:,}" for k, n in by_code["n"].items()]
undocumented = by_code.index.isin([-2, 0])

ORANGE, GREY = "#eb6834", "#8a8984"
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(labels, by_code["rate"], width=0.7,
       color=[ORANGE if u else GREY for u in undocumented])
ax.axhline(clean[Y].mean(), color="black", lw=1, ls="--")
ax.text(-0.4, clean[Y].mean() + 0.015,
        f"all customers: {clean[Y].mean():.0%}", ha="left", fontsize=9)

for i, rate in enumerate(by_code["rate"]):
    ax.text(i, rate + 0.015, f"{rate:.0%}", ha="center", fontsize=9)

ax.set_xlabel("Repayment status, September 2005 (PAY_1)   "
              "–1 = paid on time, 1–8 = months late", labelpad=8)
ax.set_ylabel("Share who defaulted in October")
ax.set_ylim(0, 0.9)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Figure 1.1: undocumented codes (orange) behave like 'paid on time'",
             loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT / "fig1-1-default-by-status.png", dpi=150)
plt.show()

## Save

The clean data goes to `data/processed/taiwan-clean.csv`. It is made by this
code and not committed. Later notebooks load it with:

```python
taiwan = pd.read_csv(ROOT / "data" / "processed" / "taiwan-clean.csv", index_col="ID")
```

In [ ]:
out = PROCESSED / "taiwan-clean.csv"
clean.to_csv(out)          # ID is written as the first column
print(out.relative_to(ROOT), clean.shape)

# Check it reads back identically
assert pd.read_csv(out, index_col="ID").equals(clean)
assert clean.shape == (29995, 24)

## Next: the shared train / validation / test split

*To be agreed by the group.* Every model is scored on the same test set, so the
split is made once, here, with a fixed seed, and saved to `data/processed/`.

## References

* Butler, O., Robinson, A. & Mulryan, L. (2026). *Data Science Toolbox Sprint 0:
  Credit Default Risk* [code and report]. GitHub, commit 577d547.
  https://github.com/Oscar339/dst-sprint0
* Yeh, I-C. (2009). *Default of Credit Card Clients* [dataset]. UCI Machine
  Learning Repository. https://doi.org/10.24432/C55S3H
* Yeh, I-C. & Lien, C-H. (2009). The comparisons of data mining techniques for
  the predictive accuracy of probability of default of credit card clients.
  *Expert Systems with Applications* 36(2), 2473–2480.
  https://doi.org/10.1016/j.eswa.2007.12.020